# Spotify ISRC + MusicBrainz で曲メタデータを補完

YouTube Music から取得した CSV に対し、Spotify で ISRC を取得して MusicBrainz と照合し、  
作曲者情報などを補完します。ISRC が取れない曲はテキスト検索にフォールバックします。

In [1]:
import json
import os
import re
import time
from pathlib import Path

import musicbrainzngs
import pandas as pd
import spotipy
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyClientCredentials
from ytmusicapi import YTMusic

In [ ]:
# MusicBrainz
musicbrainzngs.set_useragent('music_tda_tobas_sanjuanito', '0.1', 'your_email@example.com')

CACHE_DIR = Path('musicbrainz_cache')
CACHE_DIR.mkdir(exist_ok=True)

def load_json_cache(filename):
    p = CACHE_DIR / filename
    return json.loads(p.read_text(encoding='utf-8')) if p.exists() else None

def save_json_cache(filename, data):
    (CACHE_DIR / filename).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')

# Spotify
load_dotenv()
SPOTIFY_CACHE_DIR = Path('spotify_cache')
SPOTIFY_CACHE_DIR.mkdir(exist_ok=True)

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.getenv('SPOTIFY_CLIENT_ID'),
    client_secret=os.getenv('SPOTIFY_CLIENT_SECRET'),
))

# YTMusic
yt = YTMusic('browser.json')

In [3]:
_ARTICLE_RE = re.compile(
    r'^(los |las |el |la |un |una |unos |unas |de los |de las |del |de )',
    re.IGNORECASE,
)

def normalize_title(text):
    # 括弧の中の補足情報を削除
    text = re.sub(r'[\(\[\{].*?[\)\]\}]', '', text)
    text = text.lower().strip()
    text = text.replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ó', 'o').replace('ú', 'u').replace('ñ', 'n')
    return re.sub(r'\s+', ' ', (text or '').strip())

def normalize_artist(name):
    name = re.sub(r'[\(\[\{].*?[\)\]\}]', '', name)
    name = name.lower().strip()
    name = name.replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ó', 'o').replace('ú', 'u').replace('ñ', 'n')
    return _ARTICLE_RE.sub('', (name or '')).strip()

def artist_match(row_artist, matched_artist_str):
    if not matched_artist_str:
        return False
    norm_row = normalize_artist(row_artist)
    if not norm_row:
        return False
    for part in matched_artist_str.split(','):
        norm_part = normalize_artist(part.strip())
        if norm_row in norm_part or norm_part in norm_row:
            return True
    return False

def research_row_url(row):
    """ytmusicapi で title+artist を再検索し URL を補正する。マッチしなければ None。"""
    try:
        results = yt.search(f"{row['title']} {row['artist']}", filter='songs', limit=5)
        time.sleep(0.5)
    except Exception as e:
        print(f'  YTMusic error: {e}')
        return None
    norm = normalize_title(row['title'])
    for item in results:
        norm_yt = normalize_title(item.get('title', ''))
        if norm not in norm_yt and norm_yt not in norm:
            continue
        vid = item.get('videoId')
        if vid:
            new_row = row.copy()
            new_row['url'] = f'https://www.youtube.com/watch?v={vid}'
            return new_row
    return None

In [4]:
def search_recording(title, artist, limit=5):
    t = re.sub(r'\s+', ' ', (title or '').strip())
    a = re.sub(r'\s+', ' ', (artist or '').strip())
    if not t:
        return []
    safe = lambda s: re.sub(r'[^0-9a-zA-Z_-]', '_', s)
    key = f'recording_search_{safe(t)}_{safe(a)}.json'
    cached = load_json_cache(key)
    if cached is not None:
        return cached
    try:
        recs = musicbrainzngs.search_recordings(recording=t, artist=a, limit=limit).get('recording-list', [])
    except Exception as e:
        print(f'  MB search error: {t} / {a} → {e}')
        recs = []
    save_json_cache(key, recs)
    time.sleep(1.0)
    return recs


def get_work_composers(work_id):
    if not work_id:
        return []
    cached = load_json_cache(f'work_{work_id}.json')
    if cached is None:
        try:
            cached = musicbrainzngs.get_work_by_id(work_id, includes=['artist-rels'])
        except Exception as e:
            print(f'  MB work error {work_id}: {e}')
            return []
        save_json_cache(f'work_{work_id}.json', cached)
        time.sleep(1.0)
    rels = cached.get('work', {}).get('artist-relation-list', []) or []
    if isinstance(rels, dict):
        rels = [rels]
    return [
        r['artist']['name'] for r in rels
        if r.get('type', '').lower() == 'composer' and r.get('artist', {}).get('name')
    ]


def get_artist_country(artist_id):
    if not artist_id:
        return ''
    cached = load_json_cache(f'artist_{artist_id}.json')
    if cached is None:
        try:
            cached = musicbrainzngs.get_artist_by_id(artist_id, includes=['artist-rels'])
        except Exception as e:
            print(f'  MB artist error {artist_id}: {e}')
            return ''
        save_json_cache(f'artist_{artist_id}.json', cached)
        time.sleep(1.0)
    artist = cached.get('artist', {})
    # country フィールドを優先、なければ area の ISO コードで代替
    country = artist.get('country', '')
    if not country:
        codes = (artist.get('area') or {}).get('iso-3166-1-code-list', [])
        if codes:
            country = codes[0]
    return country


def get_recording_metadata(recording):
    rec_id = recording.get('id')
    if not rec_id:
        return {}
    cached = load_json_cache(f'recording_{rec_id}.json')
    if cached is None:
        try:
            cached = musicbrainzngs.get_recording_by_id(rec_id, includes=['work-rels', 'artist-rels', 'artists', 'releases'])
        except Exception as e:
            print(f'  MB recording error {rec_id}: {e}')
            return {}
        save_json_cache(f'recording_{rec_id}.json', cached)
        time.sleep(1.0)
    rec = cached.get('recording', {})

    works = rec.get('work-relation-list', []) or []
    if isinstance(works, dict):
        works = [works]
    work_ids = [w['work']['id'] for w in works if w.get('work', {}).get('id')]
    composers = sorted(set(c for wid in work_ids for c in get_work_composers(wid)))

    # キャッシュ済みdetailにartist-creditがない場合（古いキャッシュ）は入力recordingから取得
    ac = rec.get('artist-credit') or recording.get('artist-credit', []) or []
    if isinstance(ac, dict):
        ac = [ac]
    matched_artist = ', '.join(
        a.get('name') or a.get('artist', {}).get('name', '')
        for a in ac if isinstance(a, dict) and (a.get('name') or a.get('artist', {}).get('name'))
    )
    artist_id = ac[0].get('artist', {}).get('id') if ac and isinstance(ac[0], dict) else None

    return {
        'recording_id': rec_id,
        'work_ids': work_ids,
        'composer_list': composers,
        'matched_artist': matched_artist,
        'artist_id': artist_id,
    }

In [5]:
def get_isrc_from_spotify(title, artist):
    safe = lambda s: re.sub(r'[^0-9a-zA-Z_-]', '_', s)
    cache_path = SPOTIFY_CACHE_DIR / f'spotify_{safe(title)}_{safe(artist)}.json'
    if cache_path.exists():
        data = json.loads(cache_path.read_text(encoding='utf-8'))
        return data.get('isrc'), data.get('spotify_id')

    def _search(q, market=None):
        try:
            kw = dict(q=q, type='track', limit=5)
            if market:
                kw['market'] = market
            res = sp.search(**kw)
            time.sleep(0.3)
            return res.get('tracks', {}).get('items', [])
        except Exception as e:
            print(f'  Spotify error: {e}')
            return []

    norm = normalize_title(title)
    tracks = _search(f'track:"{title}" artist:"{artist}"', market='EC') or _search(f'{title} {artist}')
    chosen = next((t for t in tracks if normalize_title(t['name']) == norm), None) or (tracks[0] if tracks else None)

    isrc = chosen.get('external_ids', {}).get('isrc') if chosen else None
    spotify_id = chosen['id'] if chosen else None
    cache_path.write_text(json.dumps({'isrc': isrc, 'spotify_id': spotify_id}, ensure_ascii=False), encoding='utf-8')
    return isrc, spotify_id


def get_recording_by_isrc(isrc):
    cached = load_json_cache(f'isrc_{isrc}.json')
    if cached is not None:
        return cached
    try:
        result = musicbrainzngs.get_recordings_by_isrc(isrc)
        time.sleep(1.0)
    except musicbrainzngs.ResponseError as e:
        if '404' in str(e) or 'Not Found' in str(e):
            save_json_cache(f'isrc_{isrc}.json', {})
            return {}
        print(f'  MB ISRC error {isrc}: {e}')
        return {}
    except Exception as e:
        print(f'  MB ISRC error {isrc}: {e}')
        return {}
    save_json_cache(f'isrc_{isrc}.json', result)
    return result


def enrich_row(row):
    title, artist = row.get('title', ''), row.get('artist', '')
    out = dict(isrc='', spotify_id='', mb_recording_id='', mb_work_ids='',
               composer='', matched_artist='', artist_country='', match_method='not_found')

    isrc, spotify_id = get_isrc_from_spotify(title, artist)
    out['isrc'] = isrc or ''
    out['spotify_id'] = spotify_id or ''

    recording = None
    if isrc:
        mb = get_recording_by_isrc(isrc)
        recs = mb.get('isrc', {}).get('recording-list', [])
        if isinstance(recs, dict):
            recs = [recs]
        if recs:
            recording = recs[0]
            out['match_method'] = 'isrc'

    if recording is None:
        candidates = search_recording(title, artist, limit=3)
        if candidates:
            recording = candidates[0]
            out['match_method'] = 'text_search'

    if recording is None:
        return out

    meta = get_recording_metadata(recording)
    out['mb_recording_id'] = meta.get('recording_id', '')
    out['mb_work_ids'] = ','.join(meta.get('work_ids', []))
    out['composer'] = '; '.join(meta.get('composer_list', []))
    out['matched_artist'] = meta.get('matched_artist', '')
    out['artist_country'] = get_artist_country(meta.get('artist_id'))
    return out

In [6]:
CSV_PATHS = [
    ('sanjuanito', Path('20260418/sanjuanito_20260418_2309.csv')),
    ('tobas',      Path('20260418/tobas_20260418_2311.csv')),
]

enriched = {}  # genre → (source_path, enriched_df)

for genre, path in CSV_PATHS:
    print(f'=== {path.name} ===')
    df = pd.read_csv(path, dtype=str).fillna('')
    rows = []
    for idx, row in df.iterrows():
        rows.append({**row.to_dict(), **enrich_row(row)})
        if (idx + 1) % 20 == 0 or idx + 1 == len(df):
            counts = pd.Series([r['match_method'] for r in rows]).value_counts().to_dict()
            print(f'  {idx+1}/{len(df)} {counts}')
    enriched[genre] = (path, pd.DataFrame(rows))
    out = path.with_name(path.stem + '_isrc_enriched.csv')
    enriched[genre][1].to_csv(out, index=False, encoding='utf-8-sig')
    print(f'  → {out.name}\n')

=== sanjuanito_20260418_2309.csv ===
  MB ISRC error ushm92013180: caused by: HTTP Error 400: Bad Request
  20/200 {'text_search': 20}
  MB ISRC error ushm22088531: caused by: HTTP Error 400: Bad Request
  MB ISRC error ushm92013169: caused by: HTTP Error 400: Bad Request
  40/200 {'text_search': 28, 'isrc': 12}
  MB ISRC error usl4q1877940: caused by: HTTP Error 400: Bad Request
  MB ISRC error GB-SMU-22-08486: caused by: HTTP Error 400: Bad Request
  MB ISRC error ushm20755053: caused by: HTTP Error 400: Bad Request
  60/200 {'text_search': 46, 'isrc': 14}
  MB ISRC error ushm81397545: caused by: HTTP Error 400: Bad Request
  MB ISRC error ushm82191607: caused by: HTTP Error 400: Bad Request
  80/200 {'text_search': 66, 'isrc': 14}
  MB ISRC error ushm21853987: caused by: HTTP Error 400: Bad Request
  100/200 {'text_search': 85, 'isrc': 15}
  MB ISRC error usl4r2241316: caused by: HTTP Error 400: Bad Request
  MB ISRC error ushm22044190: caused by: HTTP Error 400: Bad Request
  MB IS

In [8]:
COUNTRY_FILTER = {'sanjuanito': ['EC', ''], 'tobas': ['BO', '']}

# tobas のみ: タイトルに含まれたら除外するジャンル・リズム名
_TOBAS_GENRE_EXCLUDE = re.compile(
    r'tinku|t\'inku|morenada|taquirari|saya|caporal|cueca|carnaval|huayno'
    r'|sikuriada|tonada|bailcito|chacarera|diablada|kullawada|llamerada'
    r'|auqui\s*auqui|waca\s*waca|kaluyo|pujllay|chuntunqui|polka|polca'
    r'|zamba|bolero|san\s*juanito|cumbia|taki\s*taki',
    re.IGNORECASE,
)

for genre, (path, df) in enriched.items():
    print(f'=== {genre} ===')

    # 国フィルタ
    df1 = df[df['artist_country'].isin(COUNTRY_FILTER[genre])]
    # selection / remix 除外
    df2 = df1[~df1['title'].str.contains(r'selection|Selección', case=False, na=False, regex=True)]
    # tobas: 他ジャンル除外
    if genre == 'tobas':
        before_genre = len(df2)
        df2 = df2[~df2['title'].apply(normalize_title).str.contains(_TOBAS_GENRE_EXCLUDE)]
        print(f'  ジャンル除外: {before_genre} → {len(df2)} 行')
    # アーティスト名寄せ確認 + URL 補正
    mask = df2.apply(lambda r: artist_match(r['artist'], r['matched_artist']), axis=1)
    matched_df, unmatched_df = df2[mask].copy(), df2[~mask]

    print(f'  {len(df)} → 国フィルタ {len(df1)} → selection除外後{len(df2)} → 一致 {len(matched_df)}, 再検索 {len(unmatched_df)}')

    recovered = [r for r in (research_row_url(row) for _, row in unmatched_df.iterrows()) if r is not None]
    final = pd.concat([matched_df, pd.DataFrame(recovered)], ignore_index=True) if recovered else matched_df

    # 重複排除: normalize済みtitle + artist + url の組み合わせで unique
    final['_t'] = final['title'].apply(normalize_title)
    final['_a'] = final['artist'].apply(normalize_artist)
    before = len(final)
    final_t_a_u = final.copy().drop_duplicates(subset=['_t', '_a', 'url']).drop(columns=['_t', '_a'])
    print(f'  重複排除: {before} → {len(final_t_a_u)} 行')

    out = path.parent / f'{genre}_filtered_unique.csv'
    final_t_a_u.to_csv(out, index=False, encoding='utf-8')
    print(f'  → {out.name} ({len(final_t_a_u)} rows)\n')

    out_title = path.parent / f'{genre}_filtered_unique_titles.csv'
    final_t = final.copy().drop_duplicates(subset=['_t']).drop(columns=['_t', '_a'])
    final_t.to_csv(out_title, index=False, encoding='utf-8', header=False)
    print(f'  → {out_title.name} ({len(final_t)} rows)\n')


=== sanjuanito ===
  200 → 国フィルタ 125 → selection除外後124 → 一致 52, 再検索 72
  重複排除: 124 → 121 行
  → sanjuanito_filtered_unique.csv (121 rows)

  → sanjuanito_filtered_unique_titles.csv (100 rows)

=== tobas ===
  ジャンル除外: 123 → 117 行
  200 → 国フィルタ 127 → selection除外後117 → 一致 37, 再検索 80
  重複排除: 117 → 115 行
  → tobas_filtered_unique.csv (115 rows)

  → tobas_filtered_unique_titles.csv (105 rows)

